
<h1>PHASE 1 — Enhanced Data Simulation</h1>
<h2>Insurance Portfolio Intelligence Suite</h2>

WHAT WE ADDED VS ORIGINAL:
- Customer Age (affects risk profile)
- City Tier (1 = metro, 2 = mid-size, 3 = small town)
- Vehicle Category (Hatchback, Sedan, SUV, Luxury)
- Risk-based Premium (not just tenure * 100)
- Variable Claim Amounts (log-normal distribution, not fixed 10000)
- Fraud Flags (2% of claims marked suspicious)


In [1]:
import pandas as pd
import numpy as np
 
# Fix random seed so results are reproducible every time you run
np.random.seed(42)
 
# ─────────────────────────────────────────
# STEP 1 — POLICY / CUSTOMER DATA
# ─────────────────────────────────────────
 
n = 1_000_000  # 1 million customers
 
# Purchase dates spread evenly across 2024
dates = pd.date_range("2024-01-01", "2024-12-31")
 
df = pd.DataFrame({
    "Customer_ID": range(1, n + 1),
    "Vehicle_ID":  range(1, n + 1),
    "Policy_Purchase_Date": np.random.choice(dates, n)
})
 
# --- Customer Age ---
# Real insurance customers range from 18 to 70
# Younger and older drivers are statistically riskier
df["Customer_Age"] = np.random.randint(18, 71, n)
 
# --- City Tier ---
# Tier 1 = Metro (Mumbai, Delhi) — higher claim frequency, higher premium
# Tier 2 = Mid-size cities — moderate
# Tier 3 = Small towns — lower frequency
df["City_Tier"] = np.random.choice([1, 2, 3], n, p=[0.40, 0.35, 0.25])
 
# --- Vehicle Category ---
# Luxury cars cost more to repair → higher claims
df["Vehicle_Category"] = np.random.choice(
    ["Hatchback", "Sedan", "SUV", "Luxury"],
    n,
    p=[0.35, 0.30, 0.25, 0.10]
)
 
# --- Vehicle Value (based on category, not fixed) ---
vehicle_value_map = {
    "Hatchback": 500_000,
    "Sedan":     800_000,
    "SUV":     1_200_000,
    "Luxury":  2_500_000
}
df["Vehicle_Value"] = df["Vehicle_Category"].map(vehicle_value_map)
 
# --- Policy Tenure ---
tenures = [1, 2, 3, 4]
probs   = [0.20, 0.30, 0.40, 0.10]
df["Policy_Tenure"] = np.random.choice(tenures, n, p=probs)
 
# --- Risk-Based Premium ---
# Base = Tenure * 1000 (increased from 100 to be realistic)
# Age surcharge: young (<25) and senior (>60) pay 20% more
# City surcharge: Tier 1 = +15%, Tier 2 = +5%, Tier 3 = 0%
# Vehicle surcharge: SUV +10%, Luxury +25%
 
base_premium = df["Policy_Tenure"] * 1000
 
age_factor = np.where(
    (df["Customer_Age"] < 25) | (df["Customer_Age"] > 60), 1.20, 1.00
)
 
city_factor = df["City_Tier"].map({1: 1.15, 2: 1.05, 3: 1.00})
 
vehicle_factor = df["Vehicle_Category"].map({
    "Hatchback": 1.00,
    "Sedan":     1.05,
    "SUV":       1.10,
    "Luxury":    1.25
})
 
df["Premium"] = (base_premium * age_factor * city_factor * vehicle_factor).round(2)
 
# --- Policy Dates ---
df["Policy_Start_Date"] = df["Policy_Purchase_Date"] + pd.Timedelta(days=365)
df["Policy_End_Date"]   = df["Policy_Start_Date"] + pd.to_timedelta(
    df["Policy_Tenure"] * 365, unit="D"
)
 
print("✅ Policy dataset created")
print(f"   Shape: {df.shape}")
print(f"   Total Premium: ₹{df['Premium'].sum():,.0f}")
print(f"\n   Vehicle Category Distribution:")
print(df["Vehicle_Category"].value_counts())
print(f"\n   City Tier Distribution:")
print(df["City_Tier"].value_counts())
 

✅ Policy dataset created
   Shape: (1000000, 11)
   Total Premium: ₹2,929,306,462

   Vehicle Category Distribution:
Vehicle_Category
Hatchback    349565
Sedan        300009
SUV          250451
Luxury        99975
Name: count, dtype: int64

   City Tier Distribution:
City_Tier
1    400599
2    349918
3    249483
Name: count, dtype: int64



# STEP 2 — CLAIMS 2025


In [2]:
# Same logic as original: only vehicles bought on 7,14,21,28 of each month
df["purchase_day"] = df["Policy_Purchase_Date"].dt.day
eligible_2025 = df[df["purchase_day"].isin([7, 14, 21, 28])]
 
# 30% of eligible vehicles file a claim
claims_2025 = eligible_2025.sample(frac=0.30, random_state=42).copy()
 
# --- Variable Claim Amounts (KEY UPGRADE) ---
# Original used fixed ₹10,000
# Now we use log-normal distribution centered around 10% of vehicle value
# Log-normal is the standard distribution used in insurance actuarial work
# It produces mostly moderate claims with occasional large ones — realistic
 
claim_mean   = claims_2025["Vehicle_Value"] * 0.10   # mean = 10% of vehicle value
claim_std    = claim_mean * 0.40                      # std = 40% of mean
 
# Log-normal parameters (mu, sigma in log space)
mu    = np.log(claim_mean**2 / np.sqrt(claim_std**2 + claim_mean**2))
sigma = np.sqrt(np.log(1 + (claim_std / claim_mean)**2))
 
claims_2025["Claim_Amount"] = np.random.lognormal(mu, sigma).round(2)
claims_2025["Claim_Amount"] = claims_2025["Claim_Amount"].clip(
    lower=1000, upper=claims_2025["Vehicle_Value"]
).round(2)
 
claims_2025["Claim_Date"] = claims_2025["Policy_Start_Date"]
claims_2025["Claim_Type"] = 1
 
# --- Fraud Flags ---
# ~2% of claims are flagged as potentially fraudulent
# Fraud detection is a hot topic in insurance analytics
fraud_mask = np.random.random(len(claims_2025)) < 0.02
claims_2025["Is_Fraud_Flag"] = fraud_mask.astype(int)
 
# Fraudulent claims tend to be suspiciously high — bump them up
claims_2025.loc[claims_2025["Is_Fraud_Flag"] == 1, "Claim_Amount"] = (
    claims_2025.loc[claims_2025["Is_Fraud_Flag"] == 1, "Vehicle_Value"] * 0.90
)
 
claims_2025 = claims_2025[[
    "Customer_ID", "Vehicle_ID", "Claim_Amount",
    "Claim_Date", "Claim_Type", "Is_Fraud_Flag"
]].copy()
 
claims_2025["Claim_ID"] = range(1, len(claims_2025) + 1)
claims_2025 = claims_2025[[
    "Claim_ID", "Customer_ID", "Vehicle_ID",
    "Claim_Amount", "Claim_Date", "Claim_Type", "Is_Fraud_Flag"
]]
claims_2025 = claims_2025.reset_index(drop=True)
 
print(f"\n✅ Claims 2025 created")
print(f"   Total Claims: {len(claims_2025):,}")
print(f"   Total Claim Amount: ₹{claims_2025['Claim_Amount'].sum():,.0f}")
print(f"   Avg Claim Amount: ₹{claims_2025['Claim_Amount'].mean():,.0f}")
print(f"   Fraud Flags: {claims_2025['Is_Fraud_Flag'].sum():,}")
 
 



✅ Claims 2025 created
   Total Claims: 39,397
   Total Claim Amount: ₹4,392,452,052
   Avg Claim Amount: ₹111,492
   Fraud Flags: 776



# STEP 3 — CLAIMS 2026 (Jan–Feb)

 

In [3]:
tenure4      = df[df["Policy_Tenure"] == 4].copy()
claims_2026  = tenure4.sample(frac=0.10, random_state=42).copy()
 
claim_dates  = pd.date_range("2026-01-01", "2026-02-28")
claims_2026["Claim_Date"] = np.random.choice(claim_dates, len(claims_2026))
 
claim_mean2  = claims_2026["Vehicle_Value"] * 0.10
claim_std2   = claim_mean2 * 0.40
 
mu2    = np.log(claim_mean2**2 / np.sqrt(claim_std2**2 + claim_mean2**2))
sigma2 = np.sqrt(np.log(1 + (claim_std2 / claim_mean2)**2))
 
claims_2026["Claim_Amount"] = np.random.lognormal(mu2, sigma2).round(2)
claims_2026["Claim_Amount"] = claims_2026["Claim_Amount"].clip(
    lower=1000, upper=claims_2026["Vehicle_Value"]
).round(2)
 
claims_2026["Claim_Type"]    = 2
 
fraud_mask2 = np.random.random(len(claims_2026)) < 0.02
claims_2026["Is_Fraud_Flag"] = fraud_mask2.astype(int)
 
claims_2026.loc[claims_2026["Is_Fraud_Flag"] == 1, "Claim_Amount"] = (
    claims_2026.loc[claims_2026["Is_Fraud_Flag"] == 1, "Vehicle_Value"] * 0.90
)
 
claims_2026 = claims_2026[[
    "Customer_ID", "Vehicle_ID", "Claim_Amount",
    "Claim_Date", "Claim_Type", "Is_Fraud_Flag"
]].copy()
 
claims_2026["Claim_ID"] = range(
    len(claims_2025) + 1,
    len(claims_2025) + len(claims_2026) + 1
)
claims_2026 = claims_2026[[
    "Claim_ID", "Customer_ID", "Vehicle_ID",
    "Claim_Amount", "Claim_Date", "Claim_Type", "Is_Fraud_Flag"
]]
claims_2026 = claims_2026.reset_index(drop=True)
 
print(f"\n✅ Claims 2026 created")
print(f"   Total Claims: {len(claims_2026):,}")
print(f"   Total Claim Amount: ₹{claims_2026['Claim_Amount'].sum():,.0f}")
 


✅ Claims 2026 created
   Total Claims: 9,970
   Total Claim Amount: ₹1,104,825,090



# STEP 4 — COMBINE & EXPORT


In [4]:
claims = pd.concat([claims_2025, claims_2026]).reset_index(drop=True)
 
# Export policy with all new fields
policy_cols = [
    "Customer_ID", "Vehicle_ID", "Customer_Age", "City_Tier",
    "Vehicle_Category", "Vehicle_Value", "Policy_Tenure",
    "Premium", "Policy_Purchase_Date", "Policy_Start_Date", "Policy_End_Date"
]
df[policy_cols].to_csv("policy_enhanced.csv", index=False)
claims.to_csv("claims_enhanced.csv", index=False)
 
print(f"\n✅ Files exported:")
print(f"   policy_enhanced.csv — {len(df):,} rows")
print(f"   claims_enhanced.csv — {len(claims):,} rows")
print(f"\n📊 FINAL SUMMARY")
print(f"   Total Premium Collected : ₹{df['Premium'].sum():>20,.0f}")
print(f"   Total Claims Filed      : {len(claims):>20,}")
print(f"   Total Claim Amount      : ₹{claims['Claim_Amount'].sum():>20,.0f}")
print(f"   Portfolio Loss Ratio    : {claims['Claim_Amount'].sum() / df['Premium'].sum():>20.2%}")
print(f"   Fraud Flags             : {claims['Is_Fraud_Flag'].sum():>20,}")
 


✅ Files exported:
   policy_enhanced.csv — 1,000,000 rows
   claims_enhanced.csv — 49,367 rows

📊 FINAL SUMMARY
   Total Premium Collected : ₹       2,929,306,462
   Total Claims Filed      :               49,367
   Total Claim Amount      : ₹       5,497,277,142
   Portfolio Loss Ratio    :              187.66%
   Fraud Flags             :                  967
